In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

dataset = pd.read_excel('../DataSet/Antes/Consolidado_contacto_2026.xlsx')
dataset.head()

,Source.Name,RUT,INTENTOS,FECHA MEJOR GESTIÓN,HORA MEJOR GESTIÓN,CALL,ESTADO,MOTIVO,NOMBRE DE BBDD,RUT EJECUTIVO,...,BARRIDO,Base,Hora,MIN,MARCA,MODELO,AÑO,EDAD,COMUNA,FECHA NACIMIENTO
0,PT_AZ_Ene.xlsx,10014568,1,2026-01-02,2026-01-02 13:50:26.880000,AZ CALL CENTER,Cerrado,YA TIENE SEGURO,BASE AUTOS RIPLEY,12027764-2,...,Recorrido,Terminado,13,50,HAVAL,H6 ELITE 2.0 AUT,2020,61,PICHIDEGUA,1964-06-13 00:00:00
1,PT_AZ_Ene.xlsx,10043674,1,2026-01-02,2026-01-02 12:54:56.160000,AZ CALL CENTER,Cerrado,NO DESEA CARGAR EN TARJETA RIPLEY,BASE AUTOS RIPLEY,13084071-k,...,Recorrido,Terminado,12,54,GREAT WALL,M4 1.5,2017,59,TALCAHUANO,1966-10-13 00:00:00
2,PT_AZ_Ene.xlsx,10114940,2,2026-01-02,2026-01-02 11:33:05.184000,AZ CALL CENTER,Abierto,AGENDAMIENTO PERSONAL,BASE AUTOS RIPLEY,13084071-k,...,Recorrido,No Terminado,11,33,HYUNDAI,SANTA FE 2.4,2018,60,COQUIMBO,1965-08-17 00:00:00
3,PT_AZ_Ene.xlsx,10153652,1,2026-01-02,2026-01-02 11:49:23.232000,AZ CALL CENTER,Cerrado,CLIENTE VENDIO/VENDERA VEHICULO,BASE AUTOS RIPLEY,15934176-3,...,Recorrido,Terminado,11,49,SUZUKI,SWIFT GL HB 1.2,2019,68,TALCA,1957-02-23 00:00:00
4,PT_AZ_Ene.xlsx,10188557,1,2026-01-02,2026-01-02 17:48:37.440000,AZ CALL CENTER,Cerrado,YA TIENE SEGURO,BASE AUTOS RIPLEY,12255216-0,...,Recorrido,Terminado,17,48,GREAT WALL,HAVAL JOLION DCT 1.5 AUT,2025,58,PUDAHUEL,1967-06-25 00:00:00


In [2]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320435 entries, 0 to 320434
Data columns (total 26 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   Source.Name          320435 non-null  object        
 1   RUT                  320435 non-null  int64         
 2   INTENTOS             320435 non-null  int64         
 3   FECHA MEJOR GESTIÓN  320435 non-null  datetime64[ns]
 4   HORA MEJOR GESTIÓN   320435 non-null  object        
 5   CALL                 320435 non-null  object        
 6   ESTADO               320435 non-null  object        
 7   MOTIVO               320435 non-null  object        
 8   NOMBRE DE BBDD       320435 non-null  object        
 9   RUT EJECUTIVO        252003 non-null  object        
 10  NOMBRE EJECUTIVO     320434 non-null  object        
 11  categoria            320435 non-null  object        
 12  PATENTE              229847 non-null  object        
 13  CONTACTABILIDA

In [3]:
dataset.describe()

,RUT,INTENTOS,FECHA MEJOR GESTIÓN,Hora,MIN
count,3.204350e+05,320435.000000,320435,320435.000000,320435.000000
mean,1.411325e+07,6.934111,2026-03-17 08:09:56.788740608,13.243706,31.158491
min,6.454420e+05,1.000000,2026-01-02 00:00:00,8.000000,0.000000
25%,1.079936e+07,2.000000,2026-02-10 00:00:00,11.000000,17.000000
50%,1.386411e+07,5.000000,2026-03-24 00:00:00,13.000000,32.000000
75%,1.700701e+07,5.000000,2026-04-24 00:00:00,16.000000,46.000000
max,2.840914e+07,77.000000,2026-05-15 00:00:00,23.000000,59.000000
std,4.309598e+06,9.357150,NaN,2.863629,16.938756


In [4]:
# --- Revisión de duplicados y valores nulos ---

# Duplicados
print(f"Filas duplicadas: {dataset.duplicated().sum()}\n")

# Nulos por columna
nulos = dataset.isnull().sum()
pct_nulos = (nulos / len(dataset) * 100).round(2)
resumen_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': pct_nulos})
resumen_nulos = resumen_nulos[resumen_nulos['Nulos'] > 0].sort_values('Porcentaje (%)', ascending=False)

if resumen_nulos.empty:
    print("No hay valores nulos.")
else:
    display(resumen_nulos)


Filas duplicadas: 1



,Nulos,Porcentaje (%)
PATENTE,90588,28.27
RUT EJECUTIVO,68432,21.36
NOMBRE EJECUTIVO,1,0.00


In [5]:
# Muestra muestra de filas con nulos en las columnas afectadas
cols_con_nulos = resumen_nulos.index.tolist()
dataset[cols_con_nulos].head(10)


,PATENTE,RUT EJECUTIVO,NOMBRE EJECUTIVO
0,LYWV26,12027764-2,Elizabeth Mora Echeverria
1,JRHH75,13084071-k,Ma Alejandra Troncoso
2,KSGC20,13084071-k,Ma Alejandra Troncoso
3,KVHD34,15934176-3,Mariana Martinez Calvin
4,TKGB81,12255216-0,Paola Noemi Veloso
5,RVFV21,26969869-1,Aroldo Sanchez Perozo
6,LXLR25,13486037-5,Moises Muoz Leal
7,JZVD68,13084071-k,Ma Alejandra Troncoso
8,HPGD50,12255216-0,Paola Noemi Veloso
9,LTSK25,13084071-k,Ma Alejandra Troncoso


In [6]:
# Filas que tienen al menos un nulo en las columnas afectadas
mask_nulos = dataset[cols_con_nulos].isnull().any(axis=1)
dataset[mask_nulos][cols_con_nulos].head(15)


,PATENTE,RUT EJECUTIVO,NOMBRE EJECUTIVO
440,LWDV56,NaN,Usuario Maquina Vicidial
638,LWVX40,NaN,Usuario Maquina Vicidial
687,KKTG76,NaN,Usuario Maquina Vicidial
1186,NaN,104010164,Catherine Lisette Lorca Coronado
1187,NaN,76128161,Discador Recoline
1188,NaN,102254880,Paula Riveros Monroy
1189,NaN,153476861,Clara Isabel Vera Fernandez
1190,NaN,153476861,Clara Isabel Vera Fernandez
1191,NaN,180960511,Ana Elizabeth Marihuan Gonzalez
1192,NaN,76128161,Discador Recoline


In [8]:
print(list(dataset.columns))
print(f"\nShape: {dataset.shape}")
print(f"\nTipos:\n{dataset.dtypes}")


['Source.Name', 'RUT', 'INTENTOS', 'FECHA MEJOR GESTIÓN', 'HORA MEJOR GESTIÓN', 'CALL', 'ESTADO', 'MOTIVO', 'NOMBRE DE BBDD', 'RUT EJECUTIVO', 'NOMBRE EJECUTIVO', 'categoria', 'PATENTE', 'CONTACTABILIDAD', 'EFECTIVIDAD', 'VENTA', 'BARRIDO', 'Base', 'Hora', 'MIN', 'MARCA', 'MODELO', 'AÑO', 'EDAD', 'COMUNA', 'FECHA NACIMIENTO']

Shape: (320435, 26)

Tipos:
Source.Name                    object
RUT                             int64
INTENTOS                        int64
FECHA MEJOR GESTIÓN    datetime64[ns]
HORA MEJOR GESTIÓN             object
CALL                           object
ESTADO                         object
MOTIVO                         object
NOMBRE DE BBDD                 object
RUT EJECUTIVO                  object
NOMBRE EJECUTIVO               object
categoria                      object
PATENTE                        object
CONTACTABILIDAD                object
EFECTIVIDAD                    object
VENTA                          object
BARRIDO                        obje